In [0]:
# Module imports
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import uuid


###############################################
# 1. API Instellingen 
###############################################

# 1.1 Base url van Gecko API
BASE_URL = "https://api.coingecko.com/api/v3"

# 1.2 Coin list om mee te werken
DEFAULT_COINS = [
    "bitcoin",
    "ethereum",
    "solana",
    "ripple",
]


###############################################
# 2. Timestamps/incremental loads Instellingen 
###############################################

# 2.1 Timestamps voor incremental loads
CONFIG_TABLE_PATH = "/config/ingest_state"

# 2.2 Default timestamp (eerste run)
DEFAULT_START_TIMESTAMP = (datetime.utcnow() - timedelta(days=1)).isoformat()

# 2.3 Functie om laatste ingest‑timestamp op te halen
def get_last_ingest_timestamp():
    try:
        df = spark.read.format("delta").load(CONFIG_TABLE_PATH)
        return df.orderBy(F.col("updated_at").desc()).first()["last_ingest_timestamp"]
    except:
        return DEFAULT_START_TIMESTAMP

# 2.4 Functie om timestamp op te slaan
def update_last_ingest_timestamp(ts):
    df = spark.createDataFrame(
        [(ts, datetime.utcnow())],
        ["last_ingest_timestamp", "updated_at"]
    )
    df.write.format("delta").mode("append").save(CONFIG_TABLE_PATH)


###############################################
# 3. Logging instellingen
###############################################

# 3.1 Pad voor logging table
LOG_TABLE_PATH = "/logs/bronze_ingest_log"

# 3.2 Functie om logregels te schrijven
def log_ingest(run_id, coin_id, status, record_count, error_message=None):
    df = spark.createDataFrame(
        [(run_id, coin_id, status, record_count, error_message, datetime.utcnow())],
        ["run_id", "coin_id", "status", "record_count", "error_message", "timestamp"]
    )
    df.write.format("delta").mode("append").save(LOG_TABLE_PATH)

# 3.3 Run‑ID genereren
RUN_ID = str(uuid.uuid4())




